In [1]:
import os, sys
sys.path.append("../..")
from utilities import plot_wellbore, RogiDataset
from pathlib import Path
import pandas as pd
from tqdm.notebook import tqdm
from plotly import express as px
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

data_path = "../../rogii-wellbore-geology-prediction"

# TVT Prediction Architecture

## Problem Framing

The task is a **signal registration problem**. At each measured depth (MD) step along a lateral wellbore, we want to find where in the stratigraphic column the well sits — expressed as a TVT value from the typewell reference log.

The primary signal is the GR log measured along the lateral, which must be registered against the typewell GR log indexed by TVT. The typewell acts as a stratigraphic reference: it tells you what GR you should expect at each TVT position. Finding the TVT at each MD step is therefore equivalent to finding where in the typewell GR profile the lateral GR matches best.

## Target Variable

The model outputs a probability distribution (softmax) over TVT candidates at each MD step. The TVT candidates are not arbitrary bins — they are a direct lookup into the typewell, centered on the local TVT anchor:

$$\mathcal{T} = \{TVT_{\text{anchor}} - 50, \; TVT_{\text{anchor}} - 49.5, \; \ldots, \; TVT_{\text{anchor}} + 49.5, \; TVT_{\text{anchor}} + 50\}$$

Since the typewell is sampled at 0.5 TVT resolution, this gives exactly **200 candidates** per window. The candidates are a slice of the typewell array, so each candidate directly corresponds to a real typewell GR value — no discretization decisions required.

The model is trained with cross-entropy loss against the ground truth TVT bin index at each MD step. At inference, the argmax over the 200 candidates gives the predicted TVT, which is converted back to an absolute TVT value for RMSE scoring.

## Sliding Window Training

Rather than one pass per well covering the full lateral, the model is trained on sliding windows along the MD axis. For each window:

- The **local anchor** is the last known TVT just before the window starts.
- The **typewell slice** of 200 candidates is centered on that local anchor TVT.
- The **target** is delta TVT relative to the local anchor at each MD step within the window.

This means every window position in every well is a valid training example, not just the prediction zone — significantly multiplying the effective dataset size and ensuring the model sees consistent input structure regardless of where in the lateral the window falls.

At inference, multiple overlapping windows are run across the lateral, each producing TVT estimates for its region. Overlapping predictions are averaged, acting as test-time augmentation and smoothing out individual window errors.

## Architecture

The model has three components: two 1D CNN heads and a 2D CNN fusion network.

### Typewell Head

A 1D CNN that takes the 200-candidate typewell GR slice as input and outputs a dense feature map of shape $(C, 200)$. Each position encodes the local GR pattern around that TVT candidate in a learned feature space.

### Lateral Head

A 1D CNN with the same structure but separate weights, taking the GR window as input and outputting a feature map of shape $(C, N_{md})$. Each position encodes the local GR pattern around that MD step.

### Outer Product Grid

The two feature maps are combined into a 2D grid of shape $(2C, 200, N_{md})$ by broadcasting and concatenating:

$$\text{grid}[:, t, d] = [f_{\text{tw}}[:, t] \;||\; f_{\text{lat}}[:, d]]$$

Every cell $(t, d)$ holds the concatenated features for TVT candidate $t$ and MD step $d$ simultaneously. This exposes the full comparison between every TVT candidate and every MD position to the downstream network — the registration problem is laid out spatially in the grid.

### Fusion CNN

A 2D CNN that takes the $(2C, 200, N_{md})$ grid and outputs a $(1, 200, N_{md})$ score map. A softmax over the 200 TVT candidates at each MD step gives the final probability distribution. The 2D CNN learns to find the smooth path of high similarity through the grid — equivalent to learned dynamic time warping.

At 200 TVT candidates × $N_{md}$ window steps, the grid is small and memory-tractable, allowing generous batch sizes and feature channels.

## Preprocessing

- NaNs in the lateral GR are dropped and the log is resampled onto a uniform MD grid via linear interpolation.
- Both logs are normalized per well to zero mean and unit variance before input.
- The typewell GR slice of 200 candidates is extracted centered on the local anchor TVT for each window.

## Data Augmentation

- **Sliding windows**: every window position along every well is a training example, each with its own local anchor. This is the primary source of data augmentation.
- **Vertical shifts**: shift the local anchor TVT by a random constant offset, moving the typewell slice up or down. The GR signal is unchanged but the registration target shifts.
- **Stretching and compression**: warp the typewell TVT axis locally, simulating formation thickness variation between the typewell and the lateral. This teaches the model robustness to imperfect typewell-to-lateral correspondence.

In [2]:
class RogiDataset():
    def __init__(self, path=Path("../../rogii-wellbore-geology-prediction/train",)):
        self.path = path
        self.well_files={
            f.name.replace("__horizontal_well.csv",""):{"Well":f,"TypeWell":f.parent/f.name.replace("__horizontal_well.csv","__typewell.csv")}
            for f in self.path.glob("*__horizontal_well.csv")
            }
        self.keys = list(self.well_files.keys())
        self._index=0

    def __getitem__(self,key:str|int):
        if isinstance(key, int):
            key = self.keys[key]
        well_data, typewell_data = pd.read_csv(self.well_files[key]["Well"]),pd.read_csv(self.well_files[key]["TypeWell"])
        well_data, typewell_data = well_data.set_index("MD").sort_index(), typewell_data.set_index("TVT").sort_index()
        pred_start_idx = well_data.index[(~well_data.TVT_input.isna()).sum()]
        last_known_idx = well_data.index[~well_data.TVT_input.isna()][-1]
        well_data["Known"] = well_data.index<=last_known_idx
        return well_data, typewell_data

    def __len__(self):
        return len(self.keys)
    
    def __iter__(self):
        self._index=0
        return self

    def __next__(self):
        if self._index>= len(self):
            raise StopIteration
        val = self[self._index]
        self._index+=1
        return val
        

In [3]:
self = RogiDataset()


# Encoding the data
The model will take both the well and typewell logs.

## Well Log
The well log data, will be sent to the well log head of the model.
- It will be of shape (batch, channel, well_log_window_size)
    - well_log_window_size is a tunable hyperparameter. It should be large enough to capture enough GR variation to disambiguate the location from the typewell log.
- Each "sample" will be a window of gr data, anchored (at the start(left)) at any MD point along the wellbore.
- The spacing of the MD index will be 1 (the data default), and the gr log will have been interpolated to remove all nans.

## Typewell Log
The typewell log data, will be sent to the typewell log head of the model.
- shape: (batch, channel, typewell_log_window_size)
    - typewell_log_window_size is a tunable hyper parameter, this is the actual length of the window
    - The tvt span of the data within that window, will be set as another tunable hyper parameter: typewell_log_range
- Each sample, will be a window of the typewell log TVT, anchored in the middle (typewell_tvt_anchor), at the closest matching tvt
    - Since tvt index of the typewell is not always spread evenly, we need to reindex that window (bounds at )

In [6]:
well_data, typewell_data = self[0]
well_data = well_data.copy()
well_data["GR"] = well_data.GR.interpolate()
well_data = well_data.dropna(subset="GR")

In [63]:
well_log_window_size = 100
typewell_delta_tvt = 100 # +/- around the tvt anchor, defines the bounds of the tvt slice extracted from typewell
typewell_slice_size = 100

In [67]:
lateral_anchor_idx = 0
lateral_tvt_anchor = well_data.TVT.iloc[lateral_anchor_idx]
lateral_gr = well_data.GR.iloc[lateral_anchor_idx:lateral_anchor_idx+well_log_window_size].values
typewell_tvt_anchor = typewell_data.index[np.abs(typewell_data.index - lateral_tvt_anchor).argmin()]


In [ ]:
typewell_window_raw = typewell_data[np.abs(typewell_data.index-lateral_tvt_anchor)<=typewell_delta_tvt/2].GR

In [ ]:
typewell_slice = typewell_window_raw.copy().reindex(np.linspace(typewell_window_raw.index[0],typewell_window_raw.index[-1],typewell_slice_size)).interpolate()

In [62]:
(lateral_gr[:,None] @ typewell_slice.values[None,:]).shape

(100, 100)

In [56]:
np.linalg.cross(lateral_gr,typewell_slice.values[None,:].T)

ValueError: Both input arrays must be (arrays of) 3-dimensional vectors, but they are 100 and 1 dimensional instead.

In [14]:
lateral_tvt_anchor

np.float64(11236.02)

In [69]:
typewell_data.index.diff().dropna().unique()

Index([0.5], dtype='float64', name='TVT')

In [66]:
typewell_data

,GR,Geology
TVT,,
11223.95,126.11,NaN
11224.45,128.22,NaN
11224.95,128.72,NaN
11225.45,128.12,NaN
11225.95,125.29,NaN
...,...,...
11869.45,38.72,BUDA
11869.95,39.16,BUDA
11870.45,39.46,BUDA


In [20]:
typewell_data.loc[anchor_tvt-100:anchor_tvt+100]

,GR,Geology
TVT,,
11223.95,126.11,NaN
11224.45,128.22,NaN
11224.95,128.72,NaN
11225.45,128.12,NaN
11225.95,125.29,NaN
...,...,...
11333.95,89.79,NaN
11334.45,92.64,NaN
11334.95,93.39,NaN
